# Peak-frame phoneme decoding: M00–M30

Choose **Runtime → Run all**. Upload the Phase 1 output ZIP when asked. This notebook downloads only the public EnglishNA IPA-CHILDES stream and writes everything under `/content`; it does not mount Google Drive.

In [ ]:
import pathlib, shutil, subprocess, sys
REPOSITORY = 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git'
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'datasets>=3.0', 'matplotlib>=3.8', 'pandas>=2.0'], check=True)
sys.path.insert(0, str(PROJECT / 'src'))
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime, then Run all again.'
print('CUDA:', torch.cuda.get_device_name(0))
print('Repository commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import files
import zipfile
from devlm.adaptation.train import discover_checkpoints, discover_feature_table
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Upload exactly one Phase 1 ZIP.')
name, content = next(iter(uploaded.items()))
if not name.lower().endswith('.zip'): raise ValueError('Expected a ZIP archive.')
zip_path = pathlib.Path('/content') / pathlib.Path(name).name; zip_path.write_bytes(content)
CHECKPOINT_ROOT = pathlib.Path('/content/devlm_phase1_checkpoints')
if CHECKPOINT_ROOT.exists(): shutil.rmtree(CHECKPOINT_ROOT)
CHECKPOINT_ROOT.mkdir()
with zipfile.ZipFile(zip_path) as archive: archive.extractall(CHECKPOINT_ROOT)
checkpoints = discover_checkpoints(CHECKPOINT_ROOT); feature_table = discover_feature_table(CHECKPOINT_ROOT)
splits = sorted(CHECKPOINT_ROOT.rglob('session_split.json'))
if len(splits) != 1: raise ValueError(f'Expected exactly one session split, found {len(splits)}')
SESSION_SPLIT = splits[0]
print(f'Validated {len(checkpoints)} checkpoints; split={SESSION_SPLIT}')

In [ ]:
# Stream the public dataset and retain exactly the Phase 1 validation sessions.
import csv, json, math
from datasets import load_dataset
validation_keys = {(str(a), str(b)) for a, b in json.loads(SESSION_SPLIT.read_text())['validation']}
SOURCE_CSV = pathlib.Path('/content/ipa_childes_phase1_validation.csv')
fieldnames = ['corpus_id','transcript_id','id','target_child_age','ipa_transcription','processed_gloss','gloss']
rows = load_dataset('phonemetransformers/IPA-CHILDES', 'EnglishNA', split='train', streaming=True)
seen, written = set(), 0
with SOURCE_CSV.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames); writer.writeheader()
    for scanned, row in enumerate(rows, 1):
        key = (str(row.get('corpus_id')), str(row.get('transcript_id'))); age = row.get('target_child_age')
        if key in validation_keys and age is not None and math.isfinite(float(age)):
            writer.writerow({name: row.get(name, '') for name in fieldnames}); seen.add(key); written += 1
        if scanned % 100000 == 0: print(f'Scanned {scanned:,}; retained {written:,} from {len(seen):,}/{len(validation_keys):,} sessions', flush=True)
missing = validation_keys - seen
if missing: raise RuntimeError(f'Missing {len(missing)} validation sessions; first={sorted(missing)[0]}')
print(f'Ready: {written:,} utterances from {len(seen):,} sessions')

In [ ]:
OUTPUT_ROOT = pathlib.Path('/content/phoneme_decoding_probe_outputs')
if OUTPUT_ROOT.exists(): shutil.rmtree(OUTPUT_ROOT)
command = [sys.executable, '-u', '-m', 'devlm.adaptation.phoneme_decoding_probe_cli', '--source-csv', str(SOURCE_CSV), '--checkpoint-root', str(CHECKPOINT_ROOT), '--session-split', str(SESSION_SPLIT), '--feature-table', str(feature_table), '--output-root', str(OUTPUT_ROOT), '--device', 'cuda']
subprocess.run(command, check=True)
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(OUTPUT_ROOT / 'summary' / 'checkpoint_metrics.tsv', sep='\t')
display(metrics[['checkpoint_id','checkpoint_hours','test_macro_f1','test_balanced_accuracy','test_accuracy','test_top3_accuracy','test_raw_input_balanced_accuracy','test_majority_balanced_accuracy']])
from IPython.display import Image
display(Image(filename=str(OUTPUT_ROOT / 'figures' / 'decoding_trajectory.png')))
display(Image(filename=str(OUTPUT_ROOT / 'figures' / 'pca_six_checkpoint_comparison.png')))
RESULT_ZIP = pathlib.Path('/content/phoneme_decoding_probe_results.zip'); assert RESULT_ZIP.is_file()
files.download(str(RESULT_ZIP))